# 01 - Extract From Source
## Chinook Data Engineering Pipeline
### Purpose: Read metadata → Extract tables from Azure SQL → Pass to Raw Zone
 

## Step 1: Setup Parameters
All values must be passed as parameters — no hardcoding allowed

In [0]:
# Setup parameters - values passed from Job at runtime
dbutils.widgets.text("catalog_name", "workspace")
dbutils.widgets.text("schema_name", "bronze")
dbutils.widgets.text("base_path", "/Volumes/workspace/raw_zone/chinook")
dbutils.widgets.text("table_name", "all")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name  = dbutils.widgets.get("schema_name")
base_path    = dbutils.widgets.get("base_path")
table_name   = dbutils.widgets.get("table_name")

print(f"catalog_name : {catalog_name}")
print(f"schema_name  : {schema_name}")
print(f"base_path    : {base_path}")
print(f"table_name   : {table_name}")

## Step 2: Read Active Tables From Metadata
Reading pipeline_control table to get list of tables to extract

In [0]:
# Read active tables from pipeline_control metadata table
active_tables = spark.sql(f"""
    SELECT table_name, file_name
    FROM {catalog_name}.{schema_name}.pipeline_control
    WHERE active_flag = 'Y'
    ORDER BY table_name
""")

print(f"Active tables to process: {active_tables.count()}")
active_tables.show()

## Step 3: Extract Data From Azure SQL Source
Using Connection Manager to read each active table
No credentials hardcoded — using chinook_connection_catalog

In [0]:
from datetime import datetime

tables_list    = active_tables.collect()
success_tables = []
failed_tables  = []

print("Starting extraction from Azure SQL...")

for row in tables_list:
    tbl = row["table_name"]
    try:
        # Read from Azure SQL using Connection Manager - no credentials hardcoded
        df = spark.read.table(f"chinook_connection_catalog.chinook.{tbl}")
        source_count = df.count()

        # Store as temp view for validation
        df.createOrReplaceTempView(f"temp_{tbl}")

        success_tables.append(tbl)
        print(f"  {tbl} - {source_count} rows extracted")

    except Exception as e:
        failed_tables.append(tbl)
        print(f"  {tbl} - FAILED: {str(e)}")

## Step 4: Extraction Summary

In [0]:
print(f"Extraction complete")
print(f"  Successful : {len(success_tables)}")
print(f"  Failed     : {len(failed_tables)}")

if failed_tables:
    print(f"  Failed tables: {failed_tables}")
else:
    print("  All tables extracted successfully - ready for Notebook 02")

## Validation
Run the cell below to verify extraction was successful


In [0]:
# Quick spot check - verify Album table extracted correctly
print("Spot check - Album table (first 5 rows):")
spark.sql("SELECT * FROM temp_Album LIMIT 5").show()
print("Validation done.")